# Confluent Schema Registry{background-image="https://docs.confluent.io/platform/current/_images/schema-registry-ecosystem.jpg" background-size="contain" background-repeat="no-repeat" background-position="right top"}
A shared contract for Kafka records


## Data contracts{background-image="https://docs.datacontract.com/assets/images/datacontractcli-7fd19880c35babb06d925e43ffb7a612.webp" background-size="contain" background-repeat="no-repeat" background-opacity="0.4" background-color="white" background-position="right top"}

:::{.fragment style="background-color: #f9f9f9; border-left: 4px solid #007acc; padding: 1em; background-opacity: 0.8;"}
The [Open Data Contract Standard (ODCS) v3.1.0](https://docs.datacontract.com/open-data-contract-standard) describes a data contract as a platform-agnostic YAML document that makes an agreement about a data product explicit. Its sections cover more than the payload schema:

- **Fundamentals:** identity, version, status, domain, purpose, usage, and limitations;
- **Schema:** objects and properties, with business meaning and physical representation;
- **Data quality:** rules and expectations for the data;
- **People and operations:** team and roles, support/communication channels, SLAs, and infrastructure.
:::

## Example: a group chat

A group chat can publish messages as events to the Kafka topic `tap-messages`. Each record describes one message:

```json
{
  "ts": 1792348935000,
  "id": "1",
  "message": "TAP lesson starts at 15"
}
```

Where  `ts` is the event time, `id` identifies this message, and `message` is its text. 

The producer and every consumer must agree on these fields and their meanings.

:::{.callout-note .fragment} 
If a consumer expects `message`, what happens if a producer silently renames it to `text`?
:::

## Schema Registry for data contract

- Schema registry is just a part of a data contract.
- The role is to help to manage the **machine-readable schema** part of a data contract: field structure, data types, schema versions, and configured compatibility checks.
- Metadata platforms such as OpenMetadata support broader governance and complement a Schema Registry; they are not replacements for it.

## Components: Format, schema, registry

- **Avro** and **Protobuf** pair binary encodings with schemas. **JSON** is a text format; **JSON Schema** describes and validates JSON documents.
- A **schema** describes the expected structure of data in a chosen format.
- A **Schema Registry** stores and versions schemas, and can check proposed changes against compatibility rules.
- A **serializer/deserializer** turns application data into bytes and back, using the agreed format and schema.
- **Parquet** is primarily a columnar storage/analytics format, not a typical per-event Kafka message format.


## Confluent Schema Registry

**Confluent Schema Registry** [`confluentinc/schema-registry`](https://github.com/confluentinc/schema-registry)

- It was developed in the Confluent/Kafka ecosystem and provides a REST API and client integrations.
- It supports Avro, Protobuf, and JSON Schema.
- It is not an Apache Software Foundation project, and the project is not uniformly Apache-2.0 licensed: the server uses the Confluent Community License


## Role of the Schema Registry in the Kafka ecosystem

```{mermaid}
%%| echo: false
sequenceDiagram
    participant P as Producer application
    participant S as Serializer
    participant SR as Schema Registry
    participant K as Kafka topic
    participant D as Deserializer
    participant C as Consumer application
    P->>S: record and schema
    S->>SR: register or look up schema for subject
    SR-->>S: schema ID
    S->>K: serialized record with schema ID
    K->>D: record bytes and schema ID
    D->>SR: fetch schema by ID (if not cached)
    SR-->>D: schema definition
    D-->>C: deserialized record
```

:::{.callout-note .fragment}
The registry is a separate service, not a Kafka broker and not the path through which every event is delivered. Kafka still carries the records. In the Confluent wire format, the serialized record includes a schema ID that lets a reader retrieve the corresponding schema.
:::

## Lifecycle of a record using schema registry

:::: {.columns}
::: {.column width="33%"}
**Actors**

- **Schema**: the structural definition of a record.
- **Subject**: the Registry name under which a schema's versions are grouped.
- **Version**: a schema's position in one subject's evolution history.
- **Schema ID**: the identifier used to resolve registered schema information; it is not the subject version.

:::{.callout-note .fragment}
With the common `TopicNameStrategy`, subjects are named `<topic>-key` or `<topic>-value`; other naming strategies are available.
:::
:::

::: {.column width="33%"}
**Schema formats**

- **Avro:** compact binary records with an explicit schema.
- **Protobuf:** message definitions use stable field numbers.
- **JSON Schema:** describes constraints on JSON-shaped data.

:::

::: {.column width="34%"}
**Lifecycle**

1. The producer serializer gets a schema.
2. The schema is registered or found under a subject.
3. A Confluent serializer frames the schema ID with the serialized key or value.
4. The consumer deserializer uses that ID to interpret the record.

:::{.callout-note .fragment}
The payload stays in Kafka; the Registry stores schema metadata. A Kafka record has a key, value, optional headers, and timestamp; topic, partition, and offset are Kafka metadata. In Confluent serialization, the schema ID is in the serialized bytes, not a Kafka header. Kafka brokers do not resolve it.
:::
:::

::::

## Schema Evolution and Compatibility Rules 

::::{.columns}
::: {.column width="50%" .fragment}
Imagine that a producer has started writing records with a new schema, while some consumers may still use the old schema.

Compatibility asks whether readers using one schema can correctly interpret data written with another. It is a check on schema evolution, not a promise that the application's business meaning has stayed the same.
:::
:::{.column width="50%" .fragment}
![](https://developer.confluent.io/5c940ba0a60bc5e46ecbd96292695fe6/schema-compatibility.svg)
:::
::::

:::{.callout-note .fragment} 
Should we check the new schema before or after deploying the producer?
:::

## Three compatibility directions

| Policy | Reader | Data being read | In plain language |
|---|---|---|---|
| **Backward** | New schema | Old-schema data | Can new code read records already written? |
| **Forward** | Old schema | New-schema data | Can old code read records newly written? |
| **Full** | Both | Both directions | Can old and new versions interoperate? |

In Confluent Schema Registry, the default compatibility level is `BACKWARD`. The configured level can be changed globally or for an individual subject. The `BACKWARD_TRANSITIVE`, `FORWARD_TRANSITIVE`, and `FULL_TRANSITIVE` variants check against all earlier subject versions, not only the latest.

## Version 1: the chat-message event

Treat each published record as an immutable event: one message was sent at a particular time. The initial Avro schema is:

```json
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
{"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"}
  ]
}
```

The timestamp is event time, not a guarantee about when Kafka received the record. `id` is the message/event identifier; the record key is a separate Kafka field and choice.

## Version 2: add sender and attachments

The chat now needs to identify the sender and describe optional attachments. Add both fields as nullable unions with `null` defaults so a new reader can still read version 1 records:

```json
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
{"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"},
    {"name": "sender", "type": ["null", {"type": "record", "name": "Sender", "fields": [
      {"name": "id", "type": "string"},
      {"name": "displayName", "type": ["null", "string"], "default": null}
    ]}], "default": null},
    {"name": "attachments", "type": ["null", {"type": "array", "items": {"type": "record", "name": "Attachment", "fields": [
      {"name": "mediaType", "type": "string"},
      {"name": "fileName", "type": ["null", "string"], "default": null},
      {"name": "sizeBytes", "type": ["null", "long"], "default": null}
    ]}}], "default": null}
  ]
}
```

An old event has neither `sender` nor `attachments`; the defaults let the new Avro reader supply `null`. An event with no known attachment can use `null`; if the contract distinguishes “known to have none” from “not supplied,” agree whether an empty array `[]` has a different meaning.

## Not every change is safe

- Renaming a field can look like removing one field and adding another.
- Changing a field's type can make old data unreadable.
- Adding a required field may leave older records without a value.
- Keeping the same type while changing what a field *means* may pass a structural check and still break an application.

:::{.callout-note .fragment}
Compatibility is format-specific. Treat it as an important automated guardrail, then review the data's meaning and deployment plan with the teams that produce and consume it.
:::

## Example: Person versions and a safe migration

| Version | Fields |
|---|---|
| v1 | `name`, `surname` |
| v2 | `name`, `surname`, `address` |
| v3 | `name`, `surname`, `address`, `phone` |

Do not replace `name` and `surname` abruptly with `full_name`. Add `full_name` as optional (with a suitable default where the format supports it), deploy consumers that handle both representations, then update producers. Deprecate the old fields only when active readers, writers, and replay requirements allow it.


## Demo: the Schema Registry API

Follow the [Confluent Cloud Schema Registry curl quickstart](https://docs.confluent.io/cloud/current/get-started/schema-registry.html#use-curl-to-view-and-manage-schemas). The requests below use its register, list, retrieve, and compatibility-check flow, adapted to the local Docker Registry at `http://localhost:8081`.

:::{.callout-note .fragment}
We will use Confluent version of the Kafka
:::





## Step 0: Start Kafka Broker

Check whether the `tap` network exists; if not, create it:

```bash
docker network inspect tap >/dev/null 2>&1 || docker network create tap
```

Start a single-node Kafka broker with KRaft mode in background.

```bash
docker run --rm -d --name kafkaServer --network tap -p 9092:9092 \
  -e KAFKA_NODE_ID=1 \
  -e KAFKA_PROCESS_ROLES=broker,controller \
  -e KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093 \
  -e KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafkaServer:9092 \
  -e KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER \
  -e KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT \
  -e KAFKA_CONTROLLER_QUORUM_VOTERS=1@localhost:9093 \
  -e KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1 \
  -e KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1 \
  -e KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS=0 \
  apache/kafka:4.4.0
```


## Step 0 continued: create the chat topic

Create a topic for the event stream

```bash
docker exec kafkaServer /opt/kafka/bin/kafka-topics.sh \
  --create --topic group-chat-events \
  --bootstrap-server kafkaServer:9092 \
  --partitions 1 --replication-factor 1

docker exec kafkaServer /opt/kafka/bin/kafka-topics.sh \
  --describe --topic group-chat-events \
  --bootstrap-server kafkaServer:9092
```

We will use this topic in the Avro producer/consumer and downstream-tool examples. With the default topic-value naming strategy, its value subject is `group-chat-events-value`; the API walkthrough and producer use this same subject and `ChatMessage` schema.

## Step 0 continued: launch Schema Registry

Use a Confluent Platform 8.2.x image. 

The Registry connects to Kafka on the shared Docker network and stores its schema history in its own internal Kafka topic. 

```bash
docker run --rm -d --name schemaRegistry --network tap -p 8081:8081 \
  -e SCHEMA_REGISTRY_HOST_NAME=schemaRegistry \
  -e SCHEMA_REGISTRY_LISTENERS=http://0.0.0.0:8081 \
  -e SCHEMA_REGISTRY_KAFKASTORE_BOOTSTRAP_SERVERS=PLAINTEXT://kafkaServer:9092 \
  -e SCHEMA_REGISTRY_KAFKASTORE_TOPIC_REPLICATION_FACTOR=1 \
  confluentinc/cp-schema-registry:8.2.2
```

Wait for the service, then verify its REST API from the host:

```bash
docker logs -f schemaRegistry
# Once it reports that it has started, press Ctrl+C to stop following logs.
curl -i http://localhost:8081/subjects
```

A fresh Registry should return an empty subject list (`[]`). Keep both containers running for the API walkthrough.

## Step 1: inspect the Registry

Start with the catalog of registered subjects:

```bash
curl -X GET http://localhost:8081/subjects
```

On a fresh registry the response is usually `[]`.

:::{.callout-note .fragment}
Is a subject a Kafka topic? 
:::

## Step 2: register the first schema

Register the `ChatMessage` value schema under the topic's subject, `group-chat-events-value`:

```bash
schema=$(cat <<'SCHEMA'
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"}
  ]
}
SCHEMA
)

payload=$(python3 -c \
  'import json,sys; print(json.dumps({"schema":sys.stdin.read()}))' \
  <<< "$schema")

curl -X POST \
  -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data "$payload" \
  http://localhost:8081/subjects/group-chat-events-value/versions
```

Python’s standard JSON library wraps the multiline Avro schema in the Registry API’s required outer `schema` string. The response contains a schema ID, for example `{"id":1}`; the ID can differ.

## Step 3: find the subject and its versions

List subjects again, then list the versions registered for our subject:

```bash
curl -X GET http://localhost:8081/subjects
curl -X GET http://localhost:8081/subjects/group-chat-events-value/versions
```

Expected shape of the responses:

```json
["group-chat-events-value"]
[1]
```

**Point out:** subject version `1` is not a Kafka offset and is not the schema ID.

## Step 4: retrieve a registered schema

Fetch the latest version stored under the subject:

```bash
curl -X GET http://localhost:8081/subjects/group-chat-events-value/versions/latest
```

The response includes the subject, version, schema ID, and schema. The `schema` value is JSON encoded inside the response JSON, so it appears escaped.

The Registry API can also fetch a schema by ID (`GET /schemas/ids/{id}`). A schema ID is a Registry identifier; it is not the subject version.

## Step 5: check a proposed evolution

Before registering a change, test a candidate schema that adds an optional `language` field with an Avro default:

```bash
schema=$(cat <<'SCHEMA'
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"},
    {"name": "language", "type": ["null", "string"], "default": null}
  ]
}
SCHEMA
)

payload=$(python3 -c \
  'import json,sys; print(json.dumps({"schema":sys.stdin.read()}))' \
  <<< "$schema")

curl -X POST \
  -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data "$payload" \
  http://localhost:8081/compatibility/subjects/group-chat-events-value/versions/latest
```

With `BACKWARD`, this should return `{"is_compatible":true}`. The request checks compatibility against the latest schema; it does not register the candidate.

## Step 6: register the compatible version

Only after discussing the compatibility result, register the candidate under the same subject:

```bash
schema=$(cat <<'SCHEMA'
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "message", "type": "string"},
    {"name": "language", "type": ["null", "string"], "default": null}
  ]
}
SCHEMA
)

payload=$(python3 -c \
  'import json,sys; print(json.dumps({"schema":sys.stdin.read()}))' \
  <<< "$schema")

curl -X POST \
  -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data "$payload" \
  http://localhost:8081/subjects/group-chat-events-value/versions

curl -X GET \
  http://localhost:8081/subjects/group-chat-events-value/versions
```

The versions should now include `1` and `2`. Registration stores a schema version; it still does not write a record to Kafka.

## Step 7: inspect compatibility configuration

The quickstart shows how to query the global compatibility configuration:

```bash
curl -X GET http://localhost:8081/config
```

A fresh Confluent Schema Registry defaults to `BACKWARD`. Compatibility can also be configured per subject. In production, the setting is part of the data-contract decision: agree on it deliberately rather than changing it casually during a demo.

## Discuss: what if we rename `message`?

Test that candidate against the same `group-chat-events-value` subject:

```bash
schema=$(cat <<'SCHEMA'
{
  "type": "record",
  "name": "ChatMessage",
  "fields": [
    {"name": "ts", "type": "long"},
    {"name": "id", "type": "string"},
    {"name": "question", "type": "string"},
    {"name": "language", "type": ["null", "string"], "default": null}
  ]
}
SCHEMA
)

payload=$(python3 -c \
  'import json,sys; print(json.dumps({"schema":sys.stdin.read()}))' \
  <<< "$schema")

curl -X POST \
  -H "Content-Type: application/vnd.schemaregistry.v1+json" \
  --data "$payload" \
  http://localhost:8081/compatibility/subjects/group-chat-events-value/versions/latest
```

:::{.callout-note .fragment}
With `BACKWARD`, this should be incompatible: older records have `message`, not the required `question` field. The request checks compatibility; it does not register the candidate.
:::


## What have we have done so far ?

We used the Registry directly to:

1. Register a schema under a subject.
2. List subjects and schema versions.
3. Retrieve the latest schema.
4. Check a proposed schema for compatibility before registering it.
5. Inspect the compatibility policy.

:::{.callout-note .fragment}
This API walkthrough did not send a message through Kafka. The next step uses a Schema Registry-aware producer and consumer to send and read Avro records.
:::

## Step 8: produce and consume Avro records

The two Python examples use the `group-chat-events` topic and the `group-chat-events-value` subject defined above. The producer uses the same `ChatMessage` schema registered in Step 2, serializes each event with its schema ID, and sends it to Kafka. The consumer reads that ID and asks Schema Registry for the schema before decoding the record.

Build or rebuild the Python image from the `tap2027/python` directory so it includes `confluent-kafka[avro]`:

```bash
docker build --tag tap:python .
```

In separate terminals, run the consumer first and then the producer. These commands use Bash or Git Bash and the `tap` Docker network:

```bash
docker run --rm --network tap -v "$(pwd)/bin:/usr/src/app/bin" \
  -e PYTHON_APP=tapSchemaRegistryConsumer.py -e PYTHONUNBUFFERED=1 tap:python

docker run --rm --network tap -v "$(pwd)/bin:/usr/src/app/bin" \
  -e PYTHON_APP=tapSchemaRegistryProducer.py -e PYTHONUNBUFFERED=1 tap:python
```

Stop either process with Ctrl+C. The scripts default to `kafkaServer:9092` and `http://schemaRegistry:8081` inside Docker; `KAFKA_TOPIC`, `KAFKA_BOOTSTRAP_SERVERS`, `SCHEMA_REGISTRY_URL`, and `GROUP_ID` can be overridden as needed.

## Test the Avro flow: send, read, inspect

**Send a record by hand** with the Avro console producer shipped in the Registry image. The schema must match the one registered by the Python producer (plain `long` for `ts`); a different schema would register a new version, subject to compatibility. Paste the JSON line, press Enter, then Ctrl+D:

```bash
docker exec -i schemaRegistry kafka-avro-console-producer \
  --bootstrap-server kafkaServer:9092 --topic group-chat-events \
  --property schema.registry.url=http://schemaRegistry:8081 \
  --property value.schema='{"type":"record","name":"ChatMessage","fields":[{"name":"ts","type":"long"},{"name":"id","type":"string"},{"name":"message","type":"string"}]}'
```

Insert this line and press Enter
```
{"ts":1792348935000,"id":"manual-1","message":"Hello from the console"}
```

**Read it back as raw bytes and as decoded Avro:**

```bash
docker exec kafkaServer /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server kafkaServer:9092 --topic group-chat-events \
  --from-beginning --max-messages 1 | od -c | head

docker exec schemaRegistry kafka-avro-console-consumer \
  --bootstrap-server kafkaServer:9092 --topic group-chat-events \
  --from-beginning --max-messages 3 \
  --property schema.registry.url=http://schemaRegistry:8081
```

**Observe:** the raw record starts with a zero *magic byte* and a 4-byte schema ID, followed by Avro data. Then confirm the subject and the ID:

```bash
curl -s http://localhost:8081/subjects/group-chat-events-value/versions/latest
```

## Guided class discussion

A team wants to add `language` to the `ChatMessage` event. As a class, decide:

1. What should the field mean, and can it be absent?
2. What change would you propose to the schema?
3. Which direction of compatibility matters if new code must read old records?
4. What should be checked before the producer starts publishing the new version?


# Kafka Connect and Schema Registry

## 1. Kafka Connect: the converter is the bridge

Kafka Connect uses **converters** to translate between Kafka bytes and Connect's internal data model. A Registry-aware converter can serialize schema-bearing records as Avro and register or look up their schemas.

For a Confluent Platform Connect worker, the value-side settings are:

```properties
value.converter=io.confluent.connect.avro.AvroConverter
value.converter.schema.registry.url=http://schemaRegistry:8081
```

Configure the key converter separately if keys also have schemas. The Registry hostname above is the Docker-network name; from a host process use `http://localhost:8081`. The worker must have the Confluent converter plugin available.

:::{.callout-important .fragment} 
the FileStream source from the Connect lesson emits plain strings without a Connect schema. Merely switching that worker to `AvroConverter` does not make those strings schema-aware. Choose a connector that produces schema-bearing records (for example, a database connector), then inspect the Registry subjects and the topic records.
:::

## Connect lab 1: start a worker with the Avro converter

Start a Connect worker on the `tap` network. It installs the Confluent Datagen connector, a test source that generates Avro-ready records without needing an external database. The worker uses `StringConverter` for keys and `AvroConverter` (with the Registry URL) for values. Replication factor `1` is for this single-broker class setup only.

```bash
docker run --rm -d --name connect --network tap -p 8083:8083 \
  -e CONNECT_BOOTSTRAP_SERVERS=kafkaServer:9092 \
  -e CONNECT_REST_ADVERTISED_HOST_NAME=connect \
  -e CONNECT_GROUP_ID=tap-connect \
  -e CONNECT_CONFIG_STORAGE_TOPIC=_connect-configs \
  -e CONNECT_OFFSET_STORAGE_TOPIC=_connect-offsets \
  -e CONNECT_STATUS_STORAGE_TOPIC=_connect-status \
  -e CONNECT_CONFIG_STORAGE_REPLICATION_FACTOR=1 \
  -e CONNECT_OFFSET_STORAGE_REPLICATION_FACTOR=1 \
  -e CONNECT_STATUS_STORAGE_REPLICATION_FACTOR=1 \
  -e CONNECT_KEY_CONVERTER=org.apache.kafka.connect.storage.StringConverter \
  -e CONNECT_VALUE_CONVERTER=io.confluent.connect.avro.AvroConverter \
  -e CONNECT_VALUE_CONVERTER_SCHEMA_REGISTRY_URL=http://schemaRegistry:8081 \
  -e CONNECT_PLUGIN_PATH=/usr/share/java,/usr/share/confluent-hub-components \
  confluentinc/cp-kafka-connect:8.2.2 \
  bash -c "confluent-hub install --no-prompt confluentinc/kafka-connect-datagen:latest && /etc/confluent/docker/run"
```

The container needs internet access to install the plugin. Wait for the REST API, then confirm the connector class is available:

```bash
docker logs -f connect          # wait for the worker to start, then Ctrl+C
curl -s http://localhost:8083/connector-plugins | grep -i datagen
```

## Connect lab 2: generate data and verify

Create the connector. The built-in `users` template produces Avro records with several fields; this run generates 20 events, at most one per second:

```bash
curl -X POST -H "Content-Type: application/json" http://localhost:8083/connectors --data '{
  "name": "datagen-users",
  "config": {
    "connector.class": "io.confluent.kafka.connect.datagen.DatagenConnector",
    "kafka.topic": "connect-users",
    "quickstart": "users",
    "max.interval": 1000,
    "iterations": 20,
    "tasks.max": "1"
  }
}'
curl -s http://localhost:8083/connectors/datagen-users/status
```

**Check each layer:**

```bash
# 1. The converter registered a subject for the topic value
curl -s http://localhost:8081/subjects
curl -s http://localhost:8081/subjects/connect-users-value/versions/latest

# 2. Consume and decode the Avro records
docker exec schemaRegistry kafka-avro-console-consumer \
  --bootstrap-server kafkaServer:9092 --topic connect-users \
  --from-beginning --max-messages 3 \
  --property schema.registry.url=http://schemaRegistry:8081
```

**Observe:** the topic carries a schema ID and binary Avro payload, not the schema itself. The subject follows the `<topic>-value` default naming strategy. Remove the connector with `curl -X DELETE http://localhost:8083/connectors/datagen-users` when finished.

# Kafka Stream and Schema Registry

## Kafka Streams: use a Registry-aware Serde

Kafka Streams is a client library, so the application chooses the serializers and deserializers (Serdes). Apache Kafka Streams does not require Schema Registry; use a Confluent Avro Serde when the input or output records use Confluent's Avro wire format.

Add `io.confluent:kafka-streams-avro-serde` from the Confluent Maven repository, choosing a release compatible with the Kafka Streams client in the project. Then configure a generic Avro Serde and use it for the chat-event values:

```java
Map<String, String> registryConfig = Map.of(
    "schema.registry.url", "http://localhost:8081"
);
Serde<GenericRecord> avroValueSerde = new GenericAvroSerde();
avroValueSerde.configure(registryConfig, false); // false: this is a value Serde

KStream<String, GenericRecord> messages = builder.stream(
    "group-chat-events",
    Consumed.with(Serdes.String(), avroValueSerde)
);
messages.to(
    "group-chat-events-copy",
    Produced.with(Serdes.String(), avroValueSerde)
);
```

The key is a plain string here; only the value uses Avro. The input deserializer resolves the schema ID through Registry (normally from its cache), and the output serializer registers or reuses the output subject's schema. In Docker, use `http://schemaRegistry:8081` instead of `localhost`.

## Streams lab: run the copy and verify

Package the snippet in a small `main` class (like the `Pipe` example) with `application.id=streams-avro-copy`, `bootstrap.servers=kafkaServer:9092`, and the Registry URL above. Then:

1. Start the application.
2. Send data to the input topic with the Python producer (Step 8) or the console producer.
3. Verify the output and the Registry:

```bash
docker exec schemaRegistry kafka-avro-console-consumer \
  --bootstrap-server kafkaServer:9092 --topic group-chat-events-copy \
  --from-beginning --max-messages 3 \
  --property schema.registry.url=http://schemaRegistry:8081

curl -s http://localhost:8081/subjects
curl -s http://localhost:8081/subjects/group-chat-events-copy-value/versions/latest
```

**Observe:** the copy topic has its own subject, but the application copied the record unchanged, so the Registry reuses the same schema ID as the input subject. Stop the application with Ctrl+C.

# Ksqldb and Schema Registry

## Introducing ksqlDB

ksqlDB is Confluent's SQL engine for creating streams, tables, and continuous queries over Kafka topics. It runs as a separate service; it is not a feature built into Kafka brokers.

| Concept | Meaning |
|---|---|
| **STREAM** | An append-only sequence of events. |
| **TABLE** | The latest state by key, updated as events arrive. |
| **Push query** | A continuous query; `EMIT CHANGES` streams results as data changes. |

```sql
SELECT id, message
FROM chat_messages
EMIT CHANGES;
```

With `VALUE_FORMAT='AVRO'`, ksqlDB can decode Registry-backed values; Kafka still transports the records.


## ksqlDB: declare the Avro stream

ksqlDB can read and write Registry-backed Avro records using `VALUE_FORMAT='AVRO'`. Its server needs the Registry URL (`ksql.schema.registry.url`) and access to both Kafka and Schema Registry.

Start the server and open the CLI on the `tap` network. The image tags are an assumption: check that they exist for your Confluent setup (ksqlDB is also published as `confluentinc/ksqldb-server`).

```bash
docker run --rm -d --name ksqldb --network tap -p 8088:8088 \
  -e KSQL_LISTENERS=http://0.0.0.0:8088 \
  -e KSQL_BOOTSTRAP_SERVERS=kafkaServer:9092 \
  -e KSQL_KSQL_SCHEMA_REGISTRY_URL=http://schemaRegistry:8081 \
  -e KSQL_KSQL_SERVICE_ID=tap_ \
  confluentinc/ksqldb-server:0.29.0

docker run --rm -it --network tap confluentinc/ksqldb-cli:0.29.0 ksql http://ksqldb:8088
```

In the CLI, read from the existing topic from the beginning and declare the stream:

```sql
SET 'auto.offset.reset' = 'earliest';
SHOW TOPICS;

CREATE STREAM chat_messages (
  ts BIGINT,
  id VARCHAR,
  message VARCHAR
) WITH (
  KAFKA_TOPIC = 'group-chat-events',
  VALUE_FORMAT = 'AVRO'
);

DESCRIBE chat_messages;
```

The SQL declaration chooses the columns ksqlDB exposes; the Avro value is decoded using the schema found through Schema Registry.

## ksqlDB lab 1: send data and run queries

**Push query** (a continuous query). Keep it running, start the Python producer from Step 8 in another terminal, and watch new rows arrive. `LIMIT` ends the query after five rows:

```sql
SELECT id, message FROM chat_messages EMIT CHANGES LIMIT 5;
```

**Send data from SQL.** Insert into a new, ksqlDB-owned topic rather than `group-chat-events`: ksqlDB generates its own Avro schema, which would normally be a different schema from `ChatMessage`.

```sql
CREATE STREAM chat_ksql (
  ts BIGINT,
  id VARCHAR,
  message VARCHAR
) WITH (
  KAFKA_TOPIC = 'chat-ksql',
  PARTITIONS = 1,
  VALUE_FORMAT = 'AVRO'
);

INSERT INTO chat_ksql (ts, id, message) VALUES (1792348935000, 'sql-1', 'Hello from ksqlDB');
INSERT INTO chat_ksql (ts, id, message) VALUES (1792348936000, 'sql-2', 'Second event');

SELECT * FROM chat_ksql EMIT CHANGES LIMIT 2;
```

**Check the Registry from another terminal:**

```bash
curl -s http://localhost:8081/subjects
curl -s http://localhost:8081/subjects/chat-ksql-value/versions/latest
docker exec schemaRegistry kafka-avro-console-consumer \
  --bootstrap-server kafkaServer:9092 --topic chat-ksql \
  --from-beginning --max-messages 2 \
  --property schema.registry.url=http://schemaRegistry:8081
```

**Observe:** the SQL `INSERT` created the subject `chat-ksql-value`. Compare its record name with `ChatMessage`: the schema was derived by ksqlDB from the column list.

## ksqlDB lab 2: transform, aggregate, pull query

**Transform** into a new Avro stream (the query keeps running in the background):

```sql
CREATE STREAM chat_messages_upper
  WITH (VALUE_FORMAT = 'AVRO') AS
  SELECT id, UPPER(message) AS message
  FROM chat_messages
  EMIT CHANGES;

SELECT * FROM chat_messages_upper EMIT CHANGES LIMIT 3;
```

**Aggregate** into a table, then run a **pull query** (a one-time lookup by key, like a database read). Use an `id` that exists in the data, for example one produced by the Python producer:

```sql
CREATE TABLE last_message AS
  SELECT id,
         LATEST_BY_OFFSET(message) AS message,
         COUNT(*) AS n
  FROM chat_messages
  GROUP BY id;

SELECT * FROM last_message WHERE id = '1';
```

**Verify and inspect:**

```sql
SHOW STREAMS;
SHOW TABLES;
SHOW QUERIES;
DESCRIBE EXTENDED chat_messages_upper;
```

```bash
curl -s http://localhost:8081/subjects        # new subjects for the output topics
docker exec schemaRegistry kafka-avro-console-consumer \
  --bootstrap-server kafkaServer:9092 --topic CHAT_MESSAGES_UPPER \
  --from-beginning --max-messages 3 \
  --property schema.registry.url=http://schemaRegistry:8081
```

The output topic is named after the stream by default (`CHAT_MESSAGES_UPPER`). Its schema is generated by ksqlDB and is not the same thing as the source schema.

**Compatibility check:** register version 2 of `ChatMessage` (with `sender` and `attachments`) as in the evolution slides, produce a record with it, and rerun the push query. The existing stream declaration still exposes only `ts`, `id`, and `message`; discuss how to expose a new field.

**Cleanup:** `DROP TABLE last_message DELETE TOPIC;` and `DROP STREAM chat_messages_upper DELETE TOPIC;` (terminate queries first if ksqlDB asks).

## Reading historical Kafka records

```{mermaid}
flowchart TD
    R[Old Kafka record: serialized bytes and schema ID] --> D[Deserializer resolves the schema ID]
    D -->|lookup if not cached| SR[Schema Registry]
    SR -->|writer schema| D
    D --> A[Format-aware writer/reader resolution]
    A --> C[Consumer using its reader schema]
```

The stored record is not rewritten when a new schema is registered. For Avro, the writer schema describes the historical bytes; schema resolution applies it to the consumer's reader schema. Keep the referenced schemas available for replay.


## Schema Registry, metadata platforms, and contracts

| Layer | Examples and responsibilities |
|---|---|
| Schema Registry | Confluent Schema Registry, Apicurio Registry, AWS Glue Schema Registry: schema versions, compatibility checks, and serializer/deserializer integrations. |
| Metadata platform | DataHub and OpenMetadata: catalogue, discovery, ownership, documentation, and lineage across datasets and pipelines. |

These layers complement each other; DataHub and OpenMetadata are not drop-in Kafka Schema Registry replacements. A data contract can include a schema, semantic expectations, quality rules, evolution expectations, ownership, and operational commitments. Lineage describes data flow and is metadata, not necessarily a contract clause.


## Recap

- Kafka carries records; Schema Registry manages schemas and their versions.
- Confluent Schema Registry is a Confluent project, separate from Apache Kafka.
- Kafka Connect uses converters; Kafka Streams uses serializers/deserializers through Serdes; ksqlDB uses schema-aware formats such as Avro.
- In each case, Registry-aware clients use schema IDs and Registry metadata while the record payload continues to travel through Kafka.
- Compatibility helps us evolve records safely, but does not guarantee that business meaning is unchanged.

## References

- [Confluent Schema Registry source repository](https://github.com/confluentinc/schema-registry)
- [Confluent Schema Registry documentation](https://docs.confluent.io/platform/current/schema-registry/index.html)
- [Schema evolution and compatibility](https://docs.confluent.io/platform/current/schema-registry/fundamentals/schema-evolution.html)
- [Schema Registry API usage](https://docs.confluent.io/platform/current/schema-registry/develop/api.html)
- [Kafka Connect converters](https://docs.confluent.io/platform/current/schema-registry/connect.html)
- [Kafka Streams Avro Serdes](https://docs.confluent.io/platform/current/streams/developer-guide/datatypes.html)
- [ksqlDB serialization formats](https://docs.confluent.io/platform/current/ksqldb/developer-guide/serialization.html)
- [Kafka documentation](https://kafka.apache.org/documentation/)
- [Spark Structured Streaming Kafka integration](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)
- [Apicurio Registry](https://www.apicur.io/registry/)
- [AWS Glue Schema Registry](https://docs.aws.amazon.com/glue/latest/dg/schema-registry.html)
- [DataHub documentation](https://docs.datahub.com/docs/)
- [OpenMetadata documentation](https://docs.open-metadata.org/)


## Final architecture: streaming data and metadata

```{mermaid}
flowchart TD
    P[Producer] --> F[Avro / JSON Schema / Protobuf serializer]
    F <--> SR[Schema Registry]
    F --> K[Kafka]
    K --> S[Spark Structured Streaming]
    S --> DF[Typed DataFrame]
    DF --> DL[Data Lake / Iceberg]
    DF --> OL[Elasticsearch / ClickHouse]
    M[DataHub / OpenMetadata] -. schemas, ownership, lineage .-> P
    M -.-> SR
    M -.-> K
    M -.-> S
    M -.-> DL
    M -.-> OL
```
